# 03. Evaluation Summary & Comparisons

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Objective**: Collect the final numbers for the report and slides straight from the result files, so nothing is typed in by hand.

1. **Section B**: best n-gram order per tokenizer (chosen on validation), with test perplexity per token and per word.
2. **Section C**: base vs. LoRA (standard and prompt-masked) on full text, answer tokens and general English.
3. **N-gram vs. neural trade-offs** for Ankora's low-resource deployment (qualitative).

In [ ]:
import sys
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

REPORTS = REPO_ROOT / "reports"

## 1. Section B: best order per tokenizer (unified corpus)
Per-token perplexity cannot be compared across tokenizers; per-word perplexity can (same text, same number of words, and every `<unk>` pays the cost of spelling the word).

In [ ]:
ngram = json.loads((REPORTS / "results_unified_all_tokenizers.json").read_text())
rows = {}
for tok, best in ngram["best_order_by_val"].items():
    row = next(r for r in ngram["results"][tok] if r["order"] == best["order"])
    rows[tok] = {"best N (val)": best["order"], "val PPL": row["val_perplexity"], "test PPL / token": row["perplexity"],
                 "test PPL / word": row["per_word_perplexity"], "vocab": row["vocab_size"], "OOV %": row["oov_rate_pct"]}
print(ngram["dataset"], "|", ngram["smoothing"])
pd.DataFrame.from_dict(rows, orient="index")

## 2. Section C: domain adaptation

In [ ]:
lora = json.loads((REPORTS / "domain_adaptation_results.json").read_text())
print(lora["split_sizes"], "| test questions seen in train/val:", lora["test_questions_seen_in_train_or_val"])
pd.DataFrame({name: {m: r[m] for m in ("full_ppl", "answer_ppl", "wikitext_ppl")} for name, r in lora["results"].items()}).T

## 3. N-gram vs. neural trade-offs (qualitative)

In [ ]:
pd.DataFrame({
    "Attribute": ["Data Requirement", "Computational Cost", "Inference Latency", "Out-of-Vocabulary Robustness", "Syntactic Generalization"],
    "N-Gram Statistical Model": ["Low (hundreds to thousands of sentences)", "Minimal (CPU, counting)", "Fast (a few hash lookups per word)", "Requires smoothing/backoff/UNK", "Limited to strict n-token window"],
    "Neural / Transformer Model": ["High (millions of tokens without pretraining)", "High (GPU training & memory)", "Moderate to High", "Subword tokenizers (BPE/WordPiece)", "High (captures long-range semantics)"],
})